# Lab 2 — Attention in Action

**Module:** Transformer & Attention | **Duration:** 90 min | **Pair programming:** Switch roles at 45 min

**Fil Rouge:** DevAssist — AI-Powered Developer Documentation Assistant for TaskFlow

---

## Objectives

1. Implement scaled dot-product attention from scratch in NumPy
2. Visualize and interpret attention weight heatmaps
3. Demonstrate empirically that context changes model behavior
4. Connect attention mechanisms to prompt engineering strategies

**Core principle:** *Attention is how the model decides which parts of your prompt matter for generating each output token. It is not magic — it is weighted averaging over context.*

---
## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import requests
import json

print("✓ All imports successful!")

In [ ]:
# === Provided Helper Functions ===

def softmax(x, axis=-1):
    """Numerically stable softmax."""
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)


def plot_attention(weights, tokens_q, tokens_k, title="Attention Weights",
                   figsize=(8, 6), cmap="Blues", ax=None):
    """Visualize an attention matrix as a heatmap."""
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(weights, cmap=cmap, vmin=0, vmax=weights.max())
    ax.set_xticks(range(len(tokens_k)))
    ax.set_xticklabels(tokens_k, rotation=45, ha="right", fontsize=10)
    ax.set_yticks(range(len(tokens_q)))
    ax.set_yticklabels(tokens_q, fontsize=10)
    ax.set_xlabel("Keys (attending TO)", fontsize=11)
    ax.set_ylabel("Queries (attending FROM)", fontsize=11)
    ax.set_title(title, fontsize=13)
    for i in range(len(tokens_q)):
        for j in range(len(tokens_k)):
            val = weights[i, j]
            color = "white" if val > weights.max() * 0.6 else "black"
            fs = 9 if len(tokens_q) <= 10 else 7
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=fs, color=color)
    plt.colorbar(im, ax=ax, label="Attention Weight", shrink=0.8)
    plt.tight_layout()
    return ax


def generate(prompt, model="llama3.2:3b", temperature=0.3, max_tokens=200):
    """Generate a completion using Ollama."""
    try:
        response = requests.post("http://localhost:11434/api/generate",
            json={"model": model, "prompt": prompt, "stream": False,
                  "options": {"temperature": temperature, "num_predict": max_tokens}},
            timeout=60)
        return response.json()["response"]
    except Exception as e:
        return f"[Generation failed: {e}]"


def compare_outputs(prompts_dict, temperature=0.3, runs=1):
    """Generate outputs for multiple prompts and display for comparison."""
    results = {}
    for label, prompt in prompts_dict.items():
        print(f"\n{'=' * 70}")
        print(f"CONDITION: {label}")
        print(f"{'=' * 70}")
        print(f"Prompt:\n{prompt[:200]}{'...' if len(prompt) > 200 else ''}")
        print(f"{'-' * 70}")
        outputs = []
        for r in range(runs):
            output = generate(prompt, temperature=temperature)
            outputs.append(output)
            prefix = f"Run {r+1}: " if runs > 1 else "Output: "
            print(f"{prefix}\n{output}\n")
        results[label] = outputs
    return results


print("✓ Helper functions loaded.")

In [ ]:
# === Check Ollama availability ===
OLLAMA_AVAILABLE = False
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"✓ Ollama running. Models: {models}")
    OLLAMA_AVAILABLE = True
except Exception:
    print("⚠ Ollama not reachable.")
    print("  Part A works without Ollama (pure NumPy).")
    print("  Part B will use pre-computed results from data/precomputed_outputs.json")

# Load precomputed data for fallback
PRECOMPUTED = {}
try:
    with open("data/precomputed_outputs.json", "r") as f:
        PRECOMPUTED = json.load(f)
    print("✓ Pre-computed outputs loaded (available as fallback).")
except FileNotFoundError:
    print("⚠ No pre-computed outputs file found.")

---
# PART A — MINI-ATTENTION FROM SCRATCH (45 minutes)

**🟢 Driver starts here. Switch roles after Part A.**

---

## Exercise 1: Implement Scaled Dot-Product Attention (15 min) — GUIDED

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) \cdot V$$

**Steps:**
1. Compute raw attention scores: `scores = Q @ K.T`
2. Scale by √d_k: `scores = scores / np.sqrt(d_k)`
3. Apply mask (if provided): set masked positions to `-1e9`
4. Apply softmax row-wise
5. Multiply weights by V

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute scaled dot-product attention.
    
    Args:
        Q: Query matrix, shape (seq_len_q, d_k)
        K: Key matrix, shape (seq_len_k, d_k)
        V: Value matrix, shape (seq_len_k, d_v)
        mask: Optional boolean mask, shape (seq_len_q, seq_len_k).
              True = masked (should not attend).
    
    Returns:
        output: shape (seq_len_q, d_v)
        weights: shape (seq_len_q, seq_len_k)
    """
    d_k = Q.shape[-1]
    
    # TODO Step 1: Compute raw attention scores (Q · Kᵀ)
    # scores = ...
    
    # TODO Step 2: Scale by sqrt(d_k)
    # scores = scores / ...
    
    # TODO Step 3: Apply mask if provided
    # if mask is not None:
    #     scores = np.where(mask, -1e9, scores)
    
    # TODO Step 4: Apply softmax row-wise
    # weights = softmax(scores, axis=-1)
    
    # TODO Step 5: Multiply weights by V
    # output = weights @ V
    
    return output, weights

In [ ]:
# === Test with TaskFlow-themed tokens ===

tokens = ["def", "create", "_task", "("]

embeddings = np.array([
    [1.0, 0.0, 0.5, 0.2],   # "def"    — keyword
    [0.2, 0.9, 0.3, 0.1],   # "create" — verb
    [0.3, 0.8, 0.4, 0.2],   # "_task"  — noun (related to "create")
    [0.8, 0.0, 0.6, 0.9],   # "("      — structural (related to "def")
])

np.random.seed(42)
d_k = 4
W_Q = np.random.randn(4, d_k) * 0.5
W_K = np.random.randn(4, d_k) * 0.5
W_V = np.random.randn(4, d_k) * 0.5

Q = embeddings @ W_Q
K = embeddings @ W_K
V = embeddings @ W_V

output, weights = scaled_dot_product_attention(Q, K, V)

print("Attention weights:")
print(np.round(weights, 3))

# Verify
print("\nRow sums:", np.round(weights.sum(axis=1), 5))
assert np.allclose(weights.sum(axis=1), 1.0), "ERROR: Rows must sum to 1!"
print("✓ All rows sum to 1 — softmax correct!")

# Visualize
plot_attention(weights, tokens, tokens, title="Self-Attention: 'def create_task('")
plt.show()

---
## Exercise 2: Causal Mask (10 min) — GUIDED

In auto-regressive generation, token *i* can only attend to tokens 0..*i* (not future tokens).

In [ ]:
def create_causal_mask(seq_len):
    """
    Create a causal mask: position i can only attend to 0..i.
    
    Returns: Boolean mask (seq_len, seq_len). True = MASKED.
    """
    # TODO: Use np.triu to create upper-triangular True mask (k=1)
    # mask = ...
    return mask

# Test
mask = create_causal_mask(4)
print("Causal mask (True = blocked):")
print(mask)

In [ ]:
# === Compare full vs. causal attention ===

output_causal, weights_causal = scaled_dot_product_attention(Q, K, V, mask=mask)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
vmax = max(weights.max(), weights_causal.max())

for ax, w, title in [(axes[0], weights, "Full Attention (bidirectional)"),
                       (axes[1], weights_causal, "Causal Attention (auto-regressive)")]:
    im = ax.imshow(w, cmap="Blues", vmin=0, vmax=vmax)
    ax.set_xticks(range(4))
    ax.set_xticklabels(tokens, rotation=45, ha="right")
    ax.set_yticks(range(4))
    ax.set_yticklabels(tokens)
    ax.set_title(title, fontsize=12)
    for i in range(4):
        for j in range(4):
            color = "white" if w[i,j] > 0.4 else "black"
            ax.text(j, i, f"{w[i,j]:.2f}", ha="center", va="center",
                    fontsize=10, color=color)

plt.suptitle("Full vs. Causal Attention — 'def create_task('", fontsize=14)
plt.tight_layout()
plt.show()

**Observation 1:** In causal attention, which token has the broadest view? Which has the narrowest? Why does this matter for prompt design?

TODO

**Observation 2:** How does the first token's representation differ between full and causal attention?

TODO

---
## Exercise 3: Multi-Head Attention (10 min) — SEMI-GUIDED

In [ ]:
def multi_head_attention(X, num_heads, d_model, mask=None):
    """
    Compute multi-head attention.
    
    Args:
        X: Input embeddings, shape (seq_len, d_model)
        num_heads: Number of attention heads
        d_model: Embedding dimension
        mask: Optional causal mask
    
    Returns:
        output: shape (seq_len, d_model)
        all_weights: List of weight matrices (one per head)
    """
    assert d_model % num_heads == 0
    d_k = d_model // num_heads
    
    all_head_outputs = []
    all_weights = []
    
    for h in range(num_heads):
        np.random.seed(42 + h)
        # TODO: Create W_Q, W_K, W_V for this head, shape (d_model, d_k)
        # W_Q_h = np.random.randn(d_model, d_k) * 0.5
        # W_K_h = ...
        # W_V_h = ...
        
        # TODO: Compute Q, K, V
        # Q_h = X @ W_Q_h
        # K_h = ...
        # V_h = ...
        
        # TODO: Apply attention
        # head_output, head_weights = scaled_dot_product_attention(Q_h, K_h, V_h, mask)
        
        all_head_outputs.append(head_output)
        all_weights.append(head_weights)
    
    # TODO: Concatenate head outputs
    # output = np.concatenate(all_head_outputs, axis=-1)
    
    return output, all_weights

In [ ]:
# === Test: DevAssist system prompt + query ===

tokens_long = ["You", "are", "DevAssist", ",", "a", "JSON", "assistant", ".",
               "List", "TaskFlow", "endpoints", "."]
n_tokens = len(tokens_long)
d_model = 8

np.random.seed(123)
X = np.random.randn(n_tokens, d_model)

output_mh, all_weights = multi_head_attention(X, num_heads=2, d_model=d_model)

# Visualize each head
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for h, (ax, w) in enumerate(zip(axes, all_weights)):
    im = ax.imshow(w, cmap="Blues", vmin=0)
    ax.set_xticks(range(n_tokens))
    ax.set_xticklabels(tokens_long, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(n_tokens))
    ax.set_yticklabels(tokens_long, fontsize=9)
    ax.set_title(f"Head {h+1}", fontsize=13)
plt.suptitle("Multi-Head Attention — Different Heads, Different Patterns", fontsize=14)
plt.tight_layout()
plt.show()

**Observation 1:** Does one head seem more "local" (nearby tokens)? Which one?

TODO

**Observation 2:** Does any head focus on specific token types (structural like `.` and `,`)?

TODO

**Observation 3:** Why is it useful to have multiple heads instead of one?

TODO

---
## Exercise 4: Why Scaling Matters (5 min) — GUIDED

In [ ]:
tokens_demo = ["def", "get", "_tasks", "()"]
np.random.seed(99)
d = 64  # large dimension makes the effect dramatic
X_demo = np.random.randn(4, d)
W_Q_d = np.random.randn(d, d) * 0.3
W_K_d = np.random.randn(d, d) * 0.3

Q_d = X_demo @ W_Q_d
K_d = X_demo @ W_K_d

# Without scaling
weights_unscaled = softmax(Q_d @ K_d.T)
# With scaling
weights_scaled = softmax((Q_d @ K_d.T) / np.sqrt(d))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, w, title in [(axes[0], weights_unscaled, f"WITHOUT Scaling (d_k={d})"),
                       (axes[1], weights_scaled, f"WITH Scaling (÷√{d}≈{np.sqrt(d):.1f})")]:
    im = ax.imshow(w, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(4)); ax.set_xticklabels(tokens_demo, rotation=45, ha="right")
    ax.set_yticks(range(4)); ax.set_yticklabels(tokens_demo)
    ax.set_title(title)
    for i in range(4):
        for j in range(4):
            color = "white" if w[i,j] > 0.5 else "black"
            ax.text(j, i, f"{w[i,j]:.2f}", ha="center", va="center",
                    fontsize=10, color=color)
plt.suptitle("Effect of Scaling on Attention Distribution", fontsize=14)
plt.tight_layout()
plt.show()

**Question:** Which version has more "spread out" attention? Why does scaling matter for training?

TODO

---
## Exercise 5: Attention Over Few-Shot Patterns (5 min) — SEMI-GUIDED

Simulate a few-shot prompt for classifying TaskFlow commit types.

In [ ]:
tokens_fs = ["Input:", "fix", "bug", "→", "bug", "|",
             "Input:", "add", "feat", "→", "feat", "|",
             "Input:", "update", "docs", "→"]

d_fs = 4
emb_fs = np.array([
    [1.0, 0.0, 0.0, 0.0],  # Input:
    [0.0, 0.8, 0.1, 0.0],  # fix
    [0.0, 0.2, 0.0, 0.9],  # bug (content)
    [0.5, 0.0, 0.5, 0.0],  # →
    [0.0, 0.1, 0.0, 1.0],  # bug (label)
    [0.3, 0.0, 0.3, 0.0],  # |
    [1.0, 0.0, 0.0, 0.0],  # Input:
    [0.0, 0.7, 0.2, 0.0],  # add
    [0.0, 0.1, 0.0, 0.8],  # feat (content)
    [0.5, 0.0, 0.5, 0.0],  # →
    [0.0, 0.0, 0.0, 0.9],  # feat (label)
    [0.3, 0.0, 0.3, 0.0],  # |
    [1.0, 0.0, 0.0, 0.0],  # Input:
    [0.0, 0.6, 0.3, 0.0],  # update
    [0.0, 0.3, 0.0, 0.7],  # docs
    [0.5, 0.0, 0.5, 0.0],  # → (predict next!)
])

# Identity projections to see raw similarity
causal_mask_fs = create_causal_mask(len(tokens_fs))
_, weights_fs = scaled_dot_product_attention(emb_fs, emb_fs, emb_fs, mask=causal_mask_fs)

# Visualize
plt.figure(figsize=(14, 10))
plot_attention(weights_fs, tokens_fs, tokens_fs,
               title="Causal Attention over Few-Shot Commit Classification")
plt.show()

# Focus on last token
print("\nAttention for the LAST '→' (what comes next?):")
print(f"{'Token':<10} {'Weight':>8}  Bar")
print("-" * 40)
for tok, w in zip(tokens_fs, weights_fs[-1]):
    bar = "█" * int(w * 40)
    print(f"{tok:<10} {w:>8.3f}  {bar}")

**Q1:** Which tokens does the final `→` attend to most? Does this make sense for predicting a label?

TODO

**Q2:** What would change if "docs" had an embedding more similar to "bug" vs "feat"?

TODO

---
# PART B — PROMPT AS CONTEXT (35 minutes)

**🔄 Switch driver/navigator roles now.**

Now we move from toy NumPy to **real model behavior**.

> **If Ollama is unavailable:** Each experiment has a fallback cell that loads pre-computed results.

---

## Experiment 1: Context Framing (12 min) — SEMI-GUIDED

In [ ]:
question = "What are the main challenges of testing a REST API like TaskFlow?"

prompts_exp1 = {
    "no_context": question,
    "junior_dev": (
        "You are explaining concepts to a junior developer who just joined "
        "the TaskFlow team and has never written an API test before.\n\n" + question),
    "senior_architect": (
        "You are presenting at a senior software architecture conference. "
        "The audience consists of principal engineers with 15+ years of experience.\n\n"
        + question),
    "json_format": (
        "You are an API that responds only in valid JSON. "
        "Return a JSON object with a key 'challenges' containing an array of strings.\n\n"
        + question),
    "misleading": (
        "TaskFlow is widely considered poorly designed and untestable. "
        "Most engineers avoid writing tests for it entirely.\n\n"
        + question),
}

if OLLAMA_AVAILABLE:
    results_exp1 = compare_outputs(prompts_exp1, temperature=0.3)
else:
    print("Using pre-computed results for Experiment 1:\n")
    exp1_data = PRECOMPUTED.get("experiment_1_context_framing", {}).get("results", {})
    for label, output in exp1_data.items():
        print(f"{'=' * 70}")
        print(f"CONDITION: {label}")
        print(f"{'=' * 70}")
        print(f"Output:\n{output}\n")

### Experiment 1 — Analysis

**Q1: How does vocabulary level change between junior dev and senior architect?**

TODO

**Q2: Does the JSON context force structured output?**

TODO

**Q3: Does the misleading preamble influence the model? Does it push back or agree?**

TODO

**Q4: Connect to attention — how does the preamble in the context window influence output token probabilities?**

TODO

---
## Experiment 2: Few-Shot Patterns (12 min) — SEMI-GUIDED

Classify code review comments — and test what happens with **wrong labels**.

In [ ]:
test_comment = "Consider using a more descriptive variable name instead of 'x'."

prompts_exp2 = {
    "zero_shot": (
        f"Classify this code review comment as 'actionable' or 'stylistic'.\n"
        f"Comment: \"{test_comment}\"\nClassification:"),
    "2shot_terse": (
        f"Classify code review comments.\n\n"
        f"Comment: \"This function will crash if the input is null.\" → actionable\n"
        f"Comment: \"Add a blank line before the return statement.\" → stylistic\n"
        f"Comment: \"{test_comment}\" →"),
    "2shot_verbose": (
        f"Classify code review comments as either 'actionable' (affects functionality, "
        f"correctness, or security) or 'stylistic' (formatting, naming, conventions).\n\n"
        f"Comment: \"This function will crash if the input is null.\"\n"
        f"Classification: actionable\n"
        f"Reason: Null input causes a runtime crash — this is a correctness issue.\n\n"
        f"Comment: \"Add a blank line before the return statement.\"\n"
        f"Classification: stylistic\n"
        f"Reason: Blank line is a formatting preference, not a functional concern.\n\n"
        f"Comment: \"{test_comment}\"\nClassification:"),
    "2shot_WRONG_labels": (
        f"Classify code review comments.\n\n"
        f"Comment: \"This function will crash if the input is null.\" → stylistic\n"
        f"Comment: \"Add a blank line before the return statement.\" → actionable\n"
        f"Comment: \"{test_comment}\" →"),
}

if OLLAMA_AVAILABLE:
    results_exp2 = compare_outputs(prompts_exp2, temperature=0.1, runs=3)
else:
    print("Using pre-computed results for Experiment 2:\n")
    exp2_data = PRECOMPUTED.get("experiment_2_fewshot_patterns", {}).get("results", {})
    for label, data in exp2_data.items():
        print(f"{'=' * 70}")
        print(f"CONDITION: {label}")
        print(f"{'=' * 70}")
        runs = data.get("runs", [data]) if isinstance(data, dict) else [data]
        for i, run in enumerate(runs):
            print(f"Run {i+1}: {run}\n")

### Experiment 2 — Analysis

**Q1: Does zero-shot produce a single word or a paragraph?**

TODO

**Q2: Does the terse 2-shot match the `→ label` pattern?**

TODO

**Q3: Does the verbose version follow the Classification + Reason format?**

TODO

**Q4 (CRITICAL): Does the wrong-labels version follow the WRONG pattern? Is the model reasoning or pattern-matching?**

TODO

**Q5: Connect to attention — how does this show that attention matches structure without verifying truth?**

TODO

---
## Experiment 3: Information Position (11 min) — AUTONOMOUS

In [ ]:
base_question = "What programming language is best for building TaskFlow's REST API?"
injected_fact = ("According to a recent industry survey, Rust is now the most popular "
                 "language for building REST APIs, surpassing Python and JavaScript.")
filler = ("Software development involves many considerations including team size, "
          "project timeline, budget constraints, and technical requirements. ") * 3

prompts_exp3 = {
    "no_injection": base_question,
    "fact_at_start": f"{injected_fact}\n\n{filler}\n\n{base_question}",
    "fact_at_end": f"{filler}\n\n{injected_fact}\n\n{base_question}",
    "fact_before_question": (
        f"{filler}\n\nContext: {injected_fact}\n\nBased on the above, {base_question}"),
    "contradictory": (
        f"According to survey A, Python is the dominant language for REST APIs.\n"
        f"According to survey B, Rust is the dominant language for REST APIs.\n"
        f"According to survey C, Go is the dominant language for REST APIs.\n\n"
        f"Based on the surveys above, {base_question}"),
}

if OLLAMA_AVAILABLE:
    results_exp3 = compare_outputs(prompts_exp3, temperature=0.3)
else:
    print("Using pre-computed results for Experiment 3:\n")
    exp3_data = PRECOMPUTED.get("experiment_3_information_position", {}).get("results", {})
    for label, output in exp3_data.items():
        print(f"{'=' * 70}")
        print(f"CONDITION: {label}")
        print(f"{'=' * 70}")
        print(f"Output:\n{output}\n")

### Experiment 3 — Analysis

**Q1: Without injection, what does the model recommend?**

TODO

**Q2: Does the model incorporate the fabricated Rust claim? Does position matter?**

TODO

**Q3: With contradictory sources, how does the model handle conflict?**

TODO

**Q4: What are the implications for RAG system design (Module 6)?**

TODO

**Q5 (Security): If an attacker can inject text into retrieved documents, how does this connect to prompt injection (Module 7)?**

TODO

---
## Mini-Report: How Context Influences Model Behavior (1 page)

Write a short report (~300–400 words) covering:
1. How preamble/framing changes output style and content
2. How few-shot examples create patterns that attention matches
3. How information position in the context affects its influence
4. One practical implication for prompt engineering
5. One security concern related to context manipulation

Ground your explanations in the attention mechanism.

*Your mini-report:*

TODO — Write your report here

---
# WRAP-UP

## Key Takeaways

1. TODO
2. TODO
3. TODO

---

## Before You Submit

1. ✅ All code cells executed
2. ✅ All TODO markers replaced with your answers
3. ✅ All 5 heatmaps generated (Exercises 1–5)
4. ✅ 5+ written observations (Part A)
5. ✅ 3 experiments analyzed (Part B)
6. ✅ Mini-report written
7. ✅ Copy `scaled_dot_product_attention`, `create_causal_mask`, `multi_head_attention` to `utils/attention_impl.py`
8. ✅ `git add -A && git commit -m 'Lab 2 complete' && git push`

---

## Homework: Attention Mini-Proof

**Due before Module 3.** Create `attention_mini_proof.ipynb`:
- Toy attention with two different contexts (same query, different surrounding info)
- Side-by-side attention weight visualizations
- Same two prompts run on a real model, outputs recorded
- Paragraph (5–8 sentences) connecting toy patterns to real behavior

---

## Preview: Lab 3

Today you learned **HOW** the model processes context (attention). In Lab 3, you'll learn **WHY** instruction-tuned models follow instructions — and how to write prompts as engineering contracts with testable acceptance criteria.

---

*Lab 2 of 8 — DevAssist / TaskFlow Lab Series*